# Treino do YOLO - YOLOv8n vs YOLO11n

Treinando duas versões do YOLO com `ultralytics` na base `african-wildlife` (búfalo, elefante, rinoceronte e zebra) pra comparar os resultados.

In [ ]:
!pip install -q ultralytics matplotlib pillow
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'

In [ ]:
EPOCHS = 15
IMGSZ = 416

model_v8 = YOLO('yolov8n.pt')
model_v8.train(data='african-wildlife.yaml', epochs=EPOCHS, imgsz=IMGSZ, batch=16, device=DEVICE, seed=42)

model_v11 = YOLO('yolo11n.pt')
model_v11.train(data='african-wildlife.yaml', epochs=EPOCHS, imgsz=IMGSZ, batch=16, device=DEVICE, seed=42)

In [1]:
metrics_v8 = model_v8.val()
metrics_v11 = model_v11.val()

for nome, m in [('YOLOv8n', metrics_v8), ('YOLO11n', metrics_v11)]:
    print(f'{nome} -> Precisão: {m.box.mp*100:.1f}% | Recall: {m.box.mr*100:.1f}% | mAP50: {m.box.map50*100:.1f}%')

YOLOv8n -> Precisão: 89.6% | Recall: 68.7% | mAP50: 84.8%
YOLO11n -> Precisão: 87.7% | Recall: 76.1% | mAP50: 87.8%


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from ultralytics.data.utils import check_det_dataset

ds_info = check_det_dataset('african-wildlife.yaml')
val_imgs = sorted(Path(ds_info['val']).glob('*.jpg'))[:4]
fig, axes = plt.subplots(len(val_imgs), 2, figsize=(12, 4 * len(val_imgs)))

for i, img_path in enumerate(val_imgs):
    axes[i, 0].imshow(model_v8.predict(str(img_path), conf=0.25, verbose=False)[0].plot()[:, :, ::-1])
    axes[i, 0].set_title(f'YOLOv8n - {img_path.name}')
    axes[i, 0].axis('off')
    axes[i, 1].imshow(model_v11.predict(str(img_path), conf=0.25, verbose=False)[0].plot()[:, :, ::-1])
    axes[i, 1].set_title(f'YOLO11n - {img_path.name}')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()